# Train a CNN on CIFAR-10

You built convolution by hand in the browser lab. Now train a real convolutional network on CIFAR-10 (60,000 small colour photos in 10 classes) with PyTorch, then look inside it. Aim: over 75% test accuracy in about 10 minutes on a GPU.

**How to use this notebook:** run the cells from top to bottom (Shift+Enter). In each step, replace every `___` in the code cell, run it, then run the **Check** cell under it. Stuck? Expand the hidden **Solution** cell (click the ▸ arrow). For speed, turn on a GPU first: *Runtime → Change runtime type → T4 GPU*.

## Setup
Run this once.

In [ ]:
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt
TEST = os.environ.get("NB_TEST") == "1"          # used only by the automatic tests; ignore it
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
tf = T.Compose([T.ToTensor(), T.Normalize((0.49, 0.48, 0.45), (0.25, 0.24, 0.26))])
if TEST:
    train_set = torchvision.datasets.FakeData(512, (3, 32, 32), 10, transform=T.ToTensor())
    test_set = torchvision.datasets.FakeData(256, (3, 32, 32), 10, transform=T.ToTensor())
else:
    train_set = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=tf)
    test_set = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=tf)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256)
classes = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
print("device:", device, "| training images:", len(train_set))

## Step 1: Define the network

Three conv blocks (conv 3x3, ReLU, 2x2 max-pool) take the image from 32x32 to 16, 8, then 4 pixels per side, with 32, 64, 128 channels. Then flatten and two linear layers. What's the size of the flattened vector?

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv2d(3, 32, 3, padding=1), nn.Conv2d(32, 64, 3, padding=1), nn.Conv2d(64, 128, 3, padding=1)])
        self.fc1 = nn.Linear(___, 256)
        self.fc2 = nn.Linear(256, 10)
    def forward(self, x):
        for conv in self.convs:
            x = F.max_pool2d(F.relu(conv(x)), 2)
        x = x.flatten(1)
        return self.fc2(F.relu(self.fc1(x)))

model = SmallCNN().to(device)
print(model)

In [ ]:
# Check: run me after the cell above
out = model(torch.randn(2, 3, 32, 32, device=device))
assert out.shape == (2, 10), "128 channels x 4 x 4 pixels = 2048 inputs to fc1"
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv2d(3, 32, 3, padding=1), nn.Conv2d(32, 64, 3, padding=1), nn.Conv2d(64, 128, 3, padding=1)])
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)
    def forward(self, x):
        for conv in self.convs:
            x = F.max_pool2d(F.relu(conv(x)), 2)
        x = x.flatten(1)
        return self.fc2(F.relu(self.fc1(x)))

model = SmallCNN().to(device)
print(model)

## Step 2: The training loop

Every PyTorch training loop has the same five lines: forward, loss, zero the old gradients, backward, step. Fill in the last two.

In [ ]:
EPOCHS = 1 if TEST else 10
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad()
        ___
        ___
        losses.append(loss.item())
    print(f"epoch {epoch + 1}: loss {np.mean(losses[-100:]):.3f}")
plt.plot(losses); plt.xlabel("batch"); plt.ylabel("loss")

In [ ]:
# Check: run me after the cell above
assert len(losses) > 0 and np.isfinite(losses).all()
if not TEST:
    assert np.mean(losses[-100:]) < np.mean(losses[:100]), "Loss should fall during training"
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
EPOCHS = 1 if TEST else 10
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    print(f"epoch {epoch + 1}: loss {np.mean(losses[-100:]):.3f}")
plt.plot(losses); plt.xlabel("batch"); plt.ylabel("loss")

## Step 3: Test accuracy

Switch to evaluation mode, turn off gradient tracking, and count correct predictions.

In [ ]:
def accuracy(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            pred = model(x.to(device)).argmax(dim=1).cpu()
            correct += ___
            total += len(y)
    return correct / total

test_acc = accuracy(model, test_loader)
print(f"test accuracy: {test_acc:.1%}")

In [ ]:
# Check: run me after the cell above
assert 0 <= test_acc <= 1
if not TEST:
    assert test_acc > 0.75, f"Aim for over 75%; you have {test_acc:.1%}. Train a few more epochs."
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
def accuracy(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            pred = model(x.to(device)).argmax(dim=1).cpu()
            correct += (pred == y).sum().item()
            total += len(y)
    return correct / total

test_acc = accuracy(model, test_loader)
print(f"test accuracy: {test_acc:.1%}")

## Step 4: Look inside: filters and an occlusion map

First-layer filters often look like colour blobs and edge detectors. Then slide a grey square over a test image and record how much the network's confidence in the true class drops: the bright regions are what it looks at.

In [ ]:
w = model.convs[0].weight.detach().cpu()
w = (w - w.min()) / (w.max() - w.min())
fig, axes = plt.subplots(4, 8, figsize=(8, 4))
for ax, f in zip(axes.ravel(), w):
    ax.imshow(f.permute(1, 2, 0)); ax.axis("off")
plt.show()

x, y = test_set[0]
model.eval()
with torch.no_grad():
    base = F.softmax(model(x[None].to(device)), dim=1)[0, y].item()
    heat = np.zeros((8, 8))
    for i in range(8):
        for j in range(8):
            occluded = x.clone()
            occluded[:, i * 4:(i + 1) * 4, j * 4:(j + 1) * 4] = 0
            p = F.softmax(model(occluded[None].to(device)), dim=1)[0, y].item()
            heat[i, j] = ___          # how much confidence dropped
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow((x.permute(1, 2, 0) * torch.tensor([0.25, 0.24, 0.26]) + torch.tensor([0.49, 0.48, 0.45])).clamp(0, 1)); axes[0].set_title(classes[y])
axes[1].imshow(heat, cmap="hot"); axes[1].set_title("confidence drop")

In [ ]:
# Check: run me after the cell above
assert heat.shape == (8, 8) and np.isfinite(heat).all() and heat.max() <= 1
print("Correct! Save this notebook to your GitHub repo (File -> Save a copy in GitHub).")

In [ ]:
#@title Solution (click ▸ to show)
w = model.convs[0].weight.detach().cpu()
w = (w - w.min()) / (w.max() - w.min())
fig, axes = plt.subplots(4, 8, figsize=(8, 4))
for ax, f in zip(axes.ravel(), w):
    ax.imshow(f.permute(1, 2, 0)); ax.axis("off")
plt.show()

x, y = test_set[0]
model.eval()
with torch.no_grad():
    base = F.softmax(model(x[None].to(device)), dim=1)[0, y].item()
    heat = np.zeros((8, 8))
    for i in range(8):
        for j in range(8):
            occluded = x.clone()
            occluded[:, i * 4:(i + 1) * 4, j * 4:(j + 1) * 4] = 0
            p = F.softmax(model(occluded[None].to(device)), dim=1)[0, y].item()
            heat[i, j] = base - p          # how much confidence dropped
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow((x.permute(1, 2, 0) * torch.tensor([0.25, 0.24, 0.26]) + torch.tensor([0.49, 0.48, 0.45])).clamp(0, 1)); axes[0].set_title(classes[y])
axes[1].imshow(heat, cmap="hot"); axes[1].set_title("confidence drop")